In [0]:
data_source_uri = "s3://dalhussein-books/DEA-Book/datasets/school/v1/"
dataset_school = 'dbfs:/mnt/DE-Associate-Book/datasets/school'
checkpoint_path = 'dbfs:/mnt/DEA-Book/checkpoints'
dlt_path = 'dbfs:/mnt/DEA-Book/dlt'
db_name = 'DE_Associate_School'
dlt_db_name = 'school_dlt_db'
spark.conf.set(f"dataset.school", dataset_school)

In [0]:
%sql
CREATE TABLE Enrollments_target AS 
SELECT * FROM parquet.`${dataset.school}/enrollments`

In [0]:
streamDF = spark.readStream.table('hive_metastore.de_associate_school.enrollments')

In [0]:
display(streamDF)

In [0]:
streamDF.writeStream.option('checkpointLocation', '/path/to/checkpoint/dir').table('hive_metastore.de_associate_school.Enrollments_target')


Es necesario tener presente que con Spark Structured Streaming, al ejecuatarse acciones sobre los diferentes df, estas se van a estar ejecutando de manera continua, ya que el sistema asume de que son estructuras que de manera continua continuan cambiando, una simple ejecucion de un display(df) puede llegar a mantener encendido el cluster.

Se podria decir que estas structuras son para tener encendido el cluster todo el tiempo. Tambien es importante tener presente que este comportamiento tambien se puede observar con objetos SQL que son creados recursos Spark Structured Streaming

![image_1777758723387.png](./image_1777758723387.png "image_1777758723387.png")

En la imagen se puede observan como se logro crear una vista temporal desde un objeto streaming, y el segundo comando se va a quedar ejecutando, siendo una unica consulta y peor siendo una consulta de Spark SQL.

"This query does not behave like a typical SQL query. Instead of executing once and returning a set of results, it initiates a continuos stream that runs indefinitely" En el libro el autor tambien explica esto, tener mucho cuidado ya que puede incrementar los costos relacionados a la permanencia del cluster encendido.





In [0]:
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")

El anterior comando funciona para ejecutar el proceso de cracion de una vista temporal desde un streaming df, es importante tener presente que las TEMP VIEW son objetos que permiten hacer esas transiciones desde diferentes recursos y objetos, de acuerdo con el libro, el convertir esta data a una temprary view permite aplicar la mayoria de transformaciones a estos df.

# Dashboard execution actions in streaming data 

![image_1777760045945.png](./image_1777760045945.png "image_1777760045945.png")

Se puede observar que el sistema esta mostrando la cantidad de datos que estan llegando al streaming, aparece una linea continua sobre el eje x debido a que no estan ingresando mas datos en su tabla de fuente Courses, es necesario tener presente que en el libro se especifica que la data que alimenta el streaming df son los nuevos datos, tambien depende del tipo de alimentacion pero por defecto son los nuevos datos.

El autor como nombra esta serie de comandos en stremaing que se empiezan a ejecutar de manera continua, se definen como _stateful_ streaming query, y su principal caractersitica es que se ejecuta de manera continua. 


In [0]:
%sql
 SELECT *
 FROM courses_streaming_tmp_vw
 ORDER BY instructor

El codigo anterior no se puede ejecutar debido a que las estructuras de streaming tienes una serie de comandos a los cuales no se les puede aplicar, entre estos comandos se encuentran los comandos de ordenamiento:

AnalysisException: Sorting is not supported on streaming DataFrames/Datasets, unless it is on aggregated DataFrame/Dataset in Complete output mode; line 3 pos 1;
~Sort [instructor#57 ASC NULLS FIRST], true
+- ~Project [course_id#55, title#56, instructor#57, category#58, price#59]

In [0]:
(output_stream_df.writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option(
        "checkpointLocation",
        "/Volumes/processing_incremental_data_databricks_chapter_5/default/school/checkpoints/instructor_counts_py"
    )
    .table("instructor_counts_py")
    .awaitTermination()
)

En el codigo anterior se modifico la direccion en el .option Checkpointlocation y este es el resultado 

![image_1777952430860.png](./image_1777952430860.png "image_1777952430860.png")

basicamente se genera una informacion de validacion que se guarda en este volumen, se recomienda usar un subfolder, y puede no guardar nada si no hay cambios en la data, esta es la respuesta de Genie:


checkpointLocation no guarda los datos de instructor_counts_py; guarda el estado técnico del stream: offsets, commits, sources y, en tu caso, también state porque haces un groupBy(...).count(...). Es decir, sirve para que Structured Streaming sepa qué ya procesó y pueda reanudar sin duplicar trabajo. Los datos finales se escriben en la tabla instructor_counts_py, no en esa ruta.

En tu código, "/Volumes/processing_incremental_data_databricks_chapter_5/default/school" parece ser la raíz del volumen school. Lo recomendable es usar una subcarpeta dedicada, por ejemplo .../school/checkpoints/instructor_counts_py. Si no ves nada, normalmente es por una de estas razones: el stream falló antes de arrancar, ya no había datos nuevos para procesar con ese checkpoint, o estás mirando la ruta esperando ver datos de negocio en vez de archivos internos del checkpoint.

Para validarlo, revisa el contenido de esa carpeta con %fs ls /Volumes/processing_incremental_data_databricks_chapter_5/default/school/checkpoints/instructor_counts_py/. Si quieres, te puedo explicar también por qué con availableNow=True a veces parece que “no hace nada”.